# 2024101026 SMAI A1

In [15]:
%pip install numpy pandas matplotlib seaborn scipy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import erfinv
import seaborn as sns
from collections import Counter


username = 'ayush.aga'
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)
print(seed)

152369566


# 1. Automated Essay Scoring System

first reading the dataset and trying to find a starting pattern

In [20]:
df = pd.read_csv("essays.csv")

print(f"\nScore distribution:\n{df['score'].value_counts().sort_index()}")


Score distribution:
score
1    1252
2    4723
3    6280
4    3926
5     970
6     156
Name: count, dtype: int64


## 1.1 Feature Engineering

In [24]:
class EssayFeatureExtractor:
    """Extracts handcrafted statistical features from raw essay text"""

    # length-category thresholds
    SHORT_WORD_MAX = 3 #word length in characters
    MEDIUM_WORD_MAX = 6
    SHORT_SENT_MAX = 7 #sentence length in words
    MEDIUM_SENT_MAX = 20
    #large sentence and words done need a threshold btw

    def __init__(self):
        """Initialize the feature extractor."""
        pass

    def _words(self, essay: str) -> list:
        """Tokenize an essay into lowercase words"""
        words = []
        current_word = ""

        for char in essay.lower():
            if ("a" <= char <= "z") or char == "'":
                current_word += char
            else:
                if current_word:
                    words.append(current_word)
                    current_word = ""

        # add the last word if the essay does not end with punctuation/space
        if current_word:
            words.append(current_word)

        return words


    def _sentences(self, essay: str) -> list:
        """Split an essay into non-empty sentences using . ! ? as boundaries."""
        sentences = []
        current_sentence = ""

        for char in essay:
            if char in ".!?":
                if current_sentence.strip():
                    sentences.append(current_sentence.strip())
                current_sentence = ""
            else:
                current_sentence += char

        # add final sentence if there is no punctuation at the end
        if current_sentence.strip():
            sentences.append(current_sentence.strip())

        return sentences

    def extract_features(self, essay: str) -> dict:
        """Extract all handcrafted features from a single essay."""
        words = self._words(essay)
        sentences = self._sentences(essay)
        n_words = len(words)
        n_sents = len(sentences)

        word_lengths = np.array([len(w) for w in words])
        sent_lengths = np.array([len(self._words(s)) for s in sentences])
        word_counts = Counter(words)

        return {
            # length / fluency
            "num_words": n_words,
            "num_sentences": n_sents,
            "num_chars": len(essay),
            # word-length profile
            "num_short_words": int((word_lengths <= self.SHORT_WORD_MAX).sum()),
            "num_medium_words": int(((word_lengths > self.SHORT_WORD_MAX)
                                     & (word_lengths <= self.MEDIUM_WORD_MAX)).sum()),
            "num_long_words": int((word_lengths > self.MEDIUM_WORD_MAX).sum()),
            # sentence-length profile
            "num_short_sentences": int((sent_lengths <= self.SHORT_SENT_MAX).sum()),
            "num_medium_sentences": int(((sent_lengths > self.SHORT_SENT_MAX)
                                         & (sent_lengths <= self.MEDIUM_SENT_MAX)).sum()),
            "num_long_sentences": int((sent_lengths > self.MEDIUM_SENT_MAX).sum()),
            # complexity
            "avg_word_length": float(word_lengths.mean()),
            "avg_sentence_length": float(sent_lengths.mean()),
            "std_sentence_length": float(sent_lengths.std()),
            # vocabulary richness
            "type_token_ratio": len(word_counts) / n_words, #the number of type of tokens
            "hapax_ratio": (sum(1 for c in word_counts.values() if c == 1) / n_words
                            if n_words else 0.0), # the number of unique words only (occurs only once)
            # mechanics / structure
            "avg_commas_per_sentence": essay.count(",") / n_sents if n_sents else 0.0,
        }

    def extract_dataset_features(self, essays) -> pd.DataFrame:
        """Extract features for all essays; returns one row per essay."""
        return pd.DataFrame([self.extract_features(e) for e in essays])

In [25]:
# Sanity-check on one essay, then extract features for the whole dataset
extractor = EssayFeatureExtractor()

demo = extractor.extract_features(df["full_text"].iloc[0])
print("Features of essay 0:")
for name, value in demo.items():
    print(f"  {name:22s} = {value:.3f}" if isinstance(value, float) else f"  {name:22s} = {value}")

features_df = extractor.extract_dataset_features(df["full_text"])
features_df["score"] = df["score"].values

print(f"\nFeature matrix shape: {features_df.shape}")
features_df.describe().T.round(3)

Features of essay 0:
  num_words              = 498
  num_sentences          = 13
  num_chars              = 2677
  num_short_words        = 212
  num_medium_words       = 214
  num_long_words         = 72
  num_short_sentences    = 1
  num_medium_sentences   = 3
  num_long_sentences     = 9
  avg_word_length        = 4.245
  avg_sentence_length    = 38.308
  std_sentence_length    = 31.682
  type_token_ratio       = 0.446
  hapax_ratio            = 0.297
  avg_commas_per_sentence = 1.077

Feature matrix shape: (17307, 16)


,count,mean,std,min,25%,50%,75%,max
num_words,17307.0,367.058,149.997,150.000,253.000,343.000,451.000,1656.000
num_sentences,17307.0,19.879,8.835,1.000,13.000,19.000,25.000,135.000
num_chars,17307.0,2063.686,924.106,712.000,1390.500,1916.000,2531.000,20450.000
num_short_words,17307.0,149.766,62.094,45.000,103.000,139.000,184.000,792.000
num_medium_words,17307.0,148.406,61.051,34.000,102.000,139.000,183.000,708.000
num_long_words,17307.0,68.886,36.378,2.000,42.000,61.000,88.000,386.000
num_short_sentences,17307.0,1.882,2.636,0.000,0.000,1.000,3.000,93.000
num_medium_sentences,17307.0,11.276,6.593,0.000,6.000,11.000,15.000,50.000
num_long_sentences,17307.0,6.721,4.098,0.000,4.000,6.000,9.000,48.000
avg_word_length,17307.0,4.432,0.293,3.438,4.228,4.430,4.635,5.667
